<a href="https://colab.research.google.com/github/15-073-AfifaAulia/123140073_Afifa-Aulia_Eksplorasi-Mahadata/blob/main/123140073_Afifa_Aulia_Eksplorasi_Mahadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Blok 1 dan 2: Persiapan Lingkungan dan Impor **Data**

In [ ]:
!pip install pyspark  #Mengunduh komponen utama untuk bisa menjalankan komputasi spark dan juga kagglehub digunakan untuk mengunduh dataset secara langsung ke storage Colab tanpa perlu mengatur API Key secara manual.

In [8]:
import kagglehub
import os
import time
from pyspark.sql import SparkSession

# 1. Mengunduh dataset spesifik Covid-19 NLP
path = kagglehub.dataset_download("elvinagammed/covid19-fake-news-dataset-nlp")

# 2. Inisialisasi Apache Spark Mode Lokal
spark = SparkSession.builder \
    .appName("ReplikasiPaperCovid") \
    .master("local[*]") \
    .getOrCreate()

# 3. Membaca dataset CSV ke dalam DataFrame Spark
file_path = os.path.join(path, "*.csv")
df = spark.read.csv(file_path, header=True, inferSchema=True)

# 4. Membersihkan data dari nilai Null (kosong)
df_clean = df.dropna(subset=["tweet", "label"])

print("data berhasil dimuat dan dibersihkan")
df_clean.show(5, truncate=70)

Using Colab cache for faster access to the 'covid19-fake-news-dataset-nlp' dataset.
data berhasil dimuat dan dibersihkan
+---+----------------------------------------------------------------------+-----+
| id|                                                                 tweet|label|
+---+----------------------------------------------------------------------+-----+
|  1|The CDC currently reports 99031 deaths. In general the discrepancie...| real|
|  2|States reported 1121 deaths a small rise from last Tuesday. Souther...| real|
|  3|Politically Correct Woman (Almost) Uses Pandemic as Excuse Not to R...| fake|
|  4|#IndiaFightsCorona: We have 1524 #COVID testing laboratories in Ind...| real|
|  5|Populous states can generate large case counts but if you look at t...| real|
+---+----------------------------------------------------------------------+-----+
only showing top 5 rows


Blok 3: Merekayasa Fitur Teks menggunakan ML Pipeline (Sesuai Paper)

Tujuan pada blok ini adalah mengubah data teks (kalimat berita) menjadi data angka untuk bisa dipelajari oleh algoritma machine learning.

1. Memecah kalimat utuh menjadi potongan kata-kata mandiri (Tokenizer).
2. Membuang kata hubung yang tidak bermakna seperti "the" atau "is" (StopWordsRemover).
3. Mengubah sisa kata-kata tersebut menjadi skor matematis berdasarkan seberapa sering kata itu muncul (HashingTF & IDF).

Seluruh langkah tersebut digunakan menjadi satu jalur otomatis yang disebut Pipeline agar proses perubahan teks menjadi angka berjalan rapi, berurutan, dan siap dimasukkan ke dalam model klasifikasi.

In [17]:
from pyspark.ml.feature import StringIndexer, Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

# 1. StringIndexer: Mengubah label teks "real" dan "fake" menjadi angka 0.0 dan 1.0
indexer = StringIndexer(inputCol="label", outputCol="label_index", handleInvalid="skip")

# 2. Tokenizer: Memecah satu kalimat utuh menjadi list kata (array of string)
tokenizer = Tokenizer(inputCol="tweet", outputCol="words")

# 3. StopWordsRemover: Membuang kata hubung/preposisi yang tidak punya makna klasifikasi (is, am, are, the, dst)
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# 4. HashingTF & IDF (Term Frequency - Inverse Document Frequency):
# Memberikan bobot tinggi pada kata yang sering muncul di satu berita, tapi jarang di berita lain.
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=3000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# 5. Membangun Pipeline
# Pipeline akan merangkai kelima proses di atas agar dieksekusi secara berurutan
pipeline = Pipeline(stages=[indexer, tokenizer, remover, hashingTF, idf])

# Mengeksekusi Pipeline ke data yang saya gunakan
print("Mengekstrak fitur teks (NLP)")
model_pipeline = pipeline.fit(df_clean)
df_features = model_pipeline.transform(df_clean)

# 6. Membagi Data: 80% untuk Latih, 20% untuk Uji
train_data, test_data = df_features.randomSplit([0.8, 0.2], seed=42)

Mengekstrak fitur teks (NLP)


Blok 4: Inisialisasi evaluasi secara menyeluruh (Sesuai Paper)

Tujuan blok ini adalah untuk mencari algoritma mana yang paling pintar dan paling cepat dalam mendeteksi berita palsu dengan menggunakan metrik akurasi, presisi, recall,f1 score, dan waktu latih.

In [15]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Membuat empat alat ukur (evaluator) yang berbeda sesuai standar paper akademik
eval_acc = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="prediction", metricName="accuracy")
eval_prec = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="prediction", metricName="weightedPrecision")
eval_rec = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="prediction", metricName="weightedRecall")
eval_f1 = MulticlassClassificationEvaluator(labelCol="label_index", predictionCol="prediction", metricName="f1")

# Membuat fungsi cetak agar tidak perlu menulis ulang kode evaluasi untuk setiap algoritma
def latih_dan_evaluasi(model, nama_model):
    print(f"Melatih {nama_model}...")
    start_time = time.time()

    # Proses Pelatihan (Fit) dan Prediksi (Transform)
    fitted_model = model.fit(train_data)
    predictions = fitted_model.transform(test_data)

    # Proses Perhitungan Metrik
    acc = eval_acc.evaluate(predictions)
    prec = eval_prec.evaluate(predictions)
    rec = eval_rec.evaluate(predictions)
    f1 = eval_f1.evaluate(predictions)
    waktu = time.time() - start_time

    print(f"--- Hasil {nama_model} ---")
    print(f"Akurasi   : {acc*100:.2f}%")
    print(f"Presisi   : {prec*100:.2f}%")
    print(f"Recall    : {rec*100:.2f}%")
    print(f"F1-Score  : {f1*100:.2f}%")
    print(f"Waktu Latih: {waktu:.2f} detik\n")

Blok 4: Pelatihan 3 Algoritma Pembanding

Tujuan dari blok ini adalah untuk menampilkan perbandingan metrik dari ketiga model beserta lama waktu prosesnya.

In [16]:
from pyspark.ml.classification import RandomForestClassifier, DecisionTreeClassifier, LogisticRegression, NaiveBayes, LinearSVC

# 1. Inisialisasi 3 Algoritma Klasifikasi dari PySpark MLlib
rf = RandomForestClassifier(featuresCol="features", labelCol="label_index", numTrees=20)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label_index")
nb = NaiveBayes(featuresCol="features", labelCol="label_index")

print("Eksperimen pembandingan algoritma\n")

# 2. Eksekusi Pelatihan secara Berurutan (memanggil fungsi dari Blok sebelumnya)
latih_dan_evaluasi(rf, "Random Forest")
latih_dan_evaluasi(dt, "Decision Tree")
latih_dan_evaluasi(nb, "Naive Bayes")

Eksperimen pembandingan algoritma

Melatih Random Forest...
--- Hasil Random Forest ---
Akurasi   : 60.18%
Presisi   : 58.13%
Recall    : 60.18%
F1-Score  : 52.50%
Waktu Latih: 47.87 detik

Melatih Decision Tree...
--- Hasil Decision Tree ---
Akurasi   : 59.56%
Presisi   : 59.20%
Recall    : 59.56%
F1-Score  : 54.43%
Waktu Latih: 49.99 detik

Melatih Naive Bayes...
--- Hasil Naive Bayes ---
Akurasi   : 61.92%
Presisi   : 65.56%
Recall    : 61.92%
F1-Score  : 63.64%
Waktu Latih: 15.80 detik

